# SANPO Preprocessing Pipeline

Preprocess SANPO video data from GCS:
- Loads egocentric video frames + depth maps
- Runs YOLO person detection
- Tracks objects frame-to-frame  
- Calculates threat scores (proximity + velocity + TTC)

**Output:** `training_cache.pkl` — Threat metrics accumulated for training

In [ ]:
# Install dependencies
!pip install -q google-cloud-storage ultralytics torch torchvision opencv-python numpy pandas
print("✓ Dependencies installed")

In [ ]:
# Clone repo (or adjust path if already cloned)
%cd /content
!git clone https://github.com/ksrikrishnareddy/CompositePerceptionEngine.git
%cd CompositePerceptionEngine
import sys
sys.path.insert(0, '/content/CompositePerceptionEngine')
print("✓ Repo cloned")

In [ ]:
# Configure and run preprocessing pipeline
from simulation.datasets.sanpo_preprocessing_pipeline import SANPOPreprocessingPipeline
import os

output_dir = '/content/preprocessing_outputs'
os.makedirs(output_dir, exist_ok=True)

# Adjust num_sessions to control runtime
# 5 sessions = ~15 min, 10 = ~1 hour, 100 = ~4 hours
num_sessions = 10

pipeline = SANPOPreprocessingPipeline(
    sanpo_root='gs://gresearch/sanpo_dataset/v0/sanpo-real',
    output_dir=output_dir,
    device='cuda'
)

# Process specific number of sessions
all_sessions = pipeline.loader.list_sessions()
pipeline.session_ids = all_sessions[:num_sessions]

print(f'Processing {len(pipeline.session_ids)} sessions...')
pipeline.run()

print(f'\n✓ Preprocessing complete!')
print(f'Results saved to {output_dir}:')

In [ ]:
# Show output files
import os
print('Output files:')
for f in os.listdir('/content/preprocessing_outputs'):
    path = os.path.join('/content/preprocessing_outputs', f)
    size = os.path.getsize(path) / (1024**2)
    print(f'  {f}: {size:.1f} MB')